# Executive Summary: Cyclistic User Behavior Analysis

### **The Challenge**
The marketing team at Cyclistic needs to transition *Casual Riders* into *Annual Members* to ensure long-term sustainable growth. This analysis evaluates **12 months of trip data (April 2025 – March 2026)** to identify behavioral differences and inform targeted conversion strategies.

### **Key Findings**
* **The Commuter vs. The Explorer:** *Annual Members* show highly stable usage during the work week (Mon-Fri) with short, efficient trips (**12.42 mins**), indicating a utility-based commute. *Casual Riders* are "Weekend Warriors," with significant spikes on Saturday/Sunday and ride durations nearly *double* that of members (**22.59 mins**).
* **Extreme Seasonality:** Casual rider volume is highly sensitive to Chicago's weather, peaking in July/August and dropping **92.68%** in winter. Members remain more resilient, with a drop in usage of **75.14%**.
---
### **Top 3 Recommendations**
#### **1. Seasonal Membership Strategy (The "Winter Gap" Fix)**
* **The Data:** Casual rider volume drops by **92.68%** in winter, while members are more resilient (only a **75.14%** drop).
* **The Recommendation:** Launch a "Pre-Season" membership drive in late February or early March. Since members are more likely to ride year-round, converting casual riders *before* the spring surge begins will stabilize annual revenue and build a consistent user base that isn't entirely dependent on perfect weather.

#### **2. Optimize for Leisure Use Cases (The "Unlimited" Value)**
* **The Data:** Casual riders average **22.59 minutes** per trip—nearly double the **12.42 minutes** of annual members. 
* **The Recommendation:** Marketing campaigns should focus on the **"unlimited ride access"** of the annual membership. For users who enjoy long-duration weekend trips, a membership removes the "per-minute" or "per-ride" transaction stress. Positioning the membership as a way to "Explore Chicago without limits" appeals directly to the high-duration behavior seen in the casual group.

#### **3. Targeted Weekend Conversion (The "Weekend Warrior")**
* **The Data:** Casual trips spike on **Saturday and Sunday**, while member usage actually decreases during the same period.
* **The Recommendation:** Target promotions specifically for the weekend at high-traffic recreational stations. Since these users clearly value the service for weekend recreation, the marketing message should frame the annual membership not as a "commuter pass," but as a **"Weekend Adventure Pass"** that pays for itself over the course of the peak summer season.
---

# Case Study: Cyclistic Bike-Share Analysis
**Analysis Period:** April 2025 – March 2026  
**Author:** Luke Shofestall | **Published:** May 2026  

---
## **Data Limitations & Scope**
_While this dataset provides a robust view of rider behavior, several constraints were identified during the analysis:_

* **Financial Data:** Specific pricing structures, revenue per ride, and membership costs were not provided. All conclusions regarding "profitability" are based on the business task's pre-defined assumption that annual members are the primary revenue driver.
* **User Anonymity:** To maintain Privacy, the dataset does not include unique user IDs or credit card hashes. This prevents tracking whether a "Casual Rider" is a one-time tourist or a frequent local user who has not yet converted.
* **Demographics:** Information such as age, gender, and residence is unavailable, limiting the ability to segment users by life stage (e.g., students vs. retirees).
---

## I. Strategic Objective

_Analyze how annual members and casual riders use Cyclistic bikes differently to help design marketing strategies aimed at converting casual riders into annual members._

### Stakeholders
* **Lily Moreno:** Director of Marketing
* **Cyclistic Executive Team**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## II. Data Integrity & Methodology
_Establishing a clean, 12-month baseline for the 2025-2026 season._
### Data Storage & Tools
* **Storage:** Data was hosted in a **Google Cloud Storage Bucket** (`trip_data_062025_042026`) to manage the large dataset size.
* **Tools:** **BigQuery** was used for data manipulation and SQL-based cleaning. **Python** (Pandas/Seaborn) will be used for final visualizations.

### Data Cleaning & Quality Assurance
Before analysis, I performed the following integrity checks in BigQuery:

> **1. Timeline Alignment:** Identified and removed stray records from March 2025 to ensure a clean 12-month analysis (April 1, 2025 – March 31, 2026).
> 
> **2. File Integrity:** Discovered a naming error in the `202601` source file (labeled internally as 2025). Validated timestamps to confirm data belonged to 2026.
> Note: To comply with data privacy regulations, all personally identifiable information (PII) such as credit card numbers and home addresses was excluded from the original dataset by the provider.
> 
> **3. Data Pruning:** Removed **29 records** with negative or zero ride durations (where `ended_at` <= `started_at`).
> 
> **4. Null Handling:** Confirmed 0% nulls in critical columns (`ride_id`, `member_casual`). Identified that station name nulls are linked to **Electric Bikes** using dockless technology, which does not impact the core analysis.

## III. Behavioral Insights: Member versus Casual
_To prepare for visualization, I aggregated the 5.7 million records into three primary summary tables:_

1. **Annual Summary:** Comparing total trips and average ride length by user type.
2. **Day of the Week Usage:** Identifying peak usage days for members vs. casual riders.
3. **Monthly Seasonality:** Tracking trip volume fluctuations across the year.

### SQL Source for Summary Tables

### Annual Summary
```sql
SELECT 
  member_casual,
  COUNT(*) AS total_trips,
  ROUND(AVG(TIMESTAMP_DIFF(ended_at, started_at, SECOND) / 60), 2) AS avg_ride_length_mins
FROM 
  `cycle-share-case-study.cycylistic_trip_data.combined_trips`
GROUP BY 
  member_casual;
```

  

### Days of The Week
```sql
SELECT 
  member_casual,
  FORMAT_TIMESTAMP('%A', started_at) AS day_of_week,
  EXTRACT(DAYOFWEEK FROM started_at) AS day_num, 
  COUNT(*) AS total_trips,
  ROUND(AVG(TIMESTAMP_DIFF(ended_at, started_at, SECOND) / 60), 2) AS avg_ride_length_mins
FROM 
  `cycle-share-case-study.cycylistic_trip_data.combined_trips`
GROUP BY 
  member_casual, day_of_week, day_num
ORDER BY 
  member_casual, day_num;

### Monthly Seasonality
```sql
SELECT 
  member_casual,
  FORMAT_TIMESTAMP('%B', started_at) AS month_name,
  EXTRACT(MONTH FROM started_at) AS month_num,
  COUNT(*) AS total_trips
FROM 
  `cycle-share-case-study.cycylistic_trip_data.combined_trips`
GROUP BY 
  member_casual, month_name, month_num
ORDER BY 
  month_num, member_casual;

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

annual_summary = pd.read_csv('/kaggle/input/datasets/lukeshofestall/cyclistic-trip-summary-0425-0326/annual_summary.csv')
day_usage= pd.read_csv('/kaggle/input/datasets/lukeshofestall/cyclistic-trip-summary-0425-0326/day_of_the_week_usage.csv')
monthly_usage = pd.read_csv('/kaggle/input/datasets/lukeshofestall/cyclistic-trip-summary-0425-0326/monthly_usage.csv')


### 3.1 Visualize Annual Usage Trends:
_To understand the fundamental difference between our two user groups, the casual users versus the members, I compared the **total number of trips** against the **average duration** of those trips for each group._


In [ ]:
# Create a side-by-side comparison of Total Trips vs Avg Duration
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Total Trips (The 2X Volume)
sns.barplot(
    data=annual_summary, 
    x='member_casual', 
    y='total_trips', 
    hue='member_casual', 
    legend=False, 
    palette='viridis', 
    ax=ax[0]
)
ax[0].set_title('Total Annual Trips', fontsize=14)
ax[0].set_ylabel('Number of Rides (in Millions)')

# Chart 2: Average Duration (The 1/2 Time)
sns.barplot(
    data=annual_summary, 
    x='member_casual', 
    y='avg_ride_length_mins', 
    hue='member_casual', 
    legend=False, 
    palette='viridis', 
    ax=ax[1]
)
ax[1].set_title('Average Ride Duration', fontsize=14)
ax[1].set_ylabel('Minutes')

plt.tight_layout()
plt.show()

In [ ]:
print(annual_summary)

**Key Insights:**
* **Member Dominance:** Annual members complete nearly **1.6 million more trips** than casual riders, suggesting a highly loyal and habitual user base.
* **Casual Engagement:** While casual riders take fewer trips, they stay on the bikes **twice as long** (average of ~22 mins vs ~12 mins for members). 
* **The "Commuter":** The short, high-frequency nature of member trips suggests routine commuting behavior. As nearly double the ride count would suggest taking a bike to work and home again.

### 3.2 Daily Usage Trends:
_By breaking down trip volume by the day of the week, we can see exactly when each group is most active._

In [ ]:

#Define the correct order of the days
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
#Convert day_of_week to a categorical type with the specified order
day_usage['day_of_week'] = pd.Categorical(day_usage['day_of_week'], categories=days_order, ordered=True)
plt.figure(figsize=(12, 6))

sns.barplot(
    data=day_usage, 
    x='day_of_week', 
    y='total_trips', 
    hue='member_casual', 
    palette='magma'
)

plt.title('Total Trips by Day of the Week', fontsize=14)
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Number of Rides', fontsize=12)
plt.legend(title='User Type')

plt.show()

**Key Insights:**
* **The Weekend Spike:** Casual rider volume peaks significantly on **Saturday and Sunday**, confirming that their primary use case is recreational.
* **Weekday Stability:** Member usage remains extremely stable from Monday through Friday and falls off on the weekend further supporting the theory that these users rely on Cyclistic for their daily commute to work or school.

### 3.3 Seasonal Usage Trends: Year-Round Engagement
_Chicago weather significantly impacts bike-share usage. This line chart tracks how trip volume fluctuates from April 2025 through March 2026._

In [ ]:
# 1. Define the correct order of the months
months_order = ['April', 'May', 'June', 'July', 'August', 'September', 
                'October', 'November', 'December', 'January', 'February', 'March']

# 2. Convert month_name to a categorical type with the specified order
monthly_usage['month_name'] = pd.Categorical(monthly_usage['month_name'], categories=months_order, ordered=True)

# 3. Create the Line Chart
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=monthly_usage.sort_values('month_name'), 
    x='month_name', 
    y='total_trips', 
    hue='member_casual', 
    marker='o', # Adds dots to the data points
    palette='viridis'
)

plt.title('Monthly Trip Volume: Seasonality Trends', fontsize=14)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Total Trips', fontsize=12)
plt.legend(title='User Type')

plt.show()

In [ ]:
# Calculate the exact drop for Casual Riders
casual_only = monthly_usage[monthly_usage['member_casual'] == 'casual']
peak_trips = casual_only['total_trips'].max()
low_trips = casual_only['total_trips'].min()

drop_percent = ((peak_trips - low_trips) / peak_trips) * 100


print(f"Seasonal Percentage Drop (Casual): {drop_percent:.2f}%")
# Calculate the exact drop for Annual Members
member_only = monthly_usage[monthly_usage['member_casual'] == 'member']
m_peak_trips = member_only['total_trips'].max()
m_low_trips = member_only['total_trips'].min()

m_drop_percent = ((m_peak_trips - m_low_trips) / m_peak_trips) * 100


print(f"Seasonal Percentage Drop (Members): {m_drop_percent:.2f}%")

**Key Insights:**
* **Summer Peak:** Both groups peak in **July and August**, but the drop-off for casual riders is much more severe as the temperature cools and tourism slows.
* **Winter Resilience:** Annual members continue to use the service through the winter months at a much higher rate than casual riders, showing that the membership provides value even in less-than-ideal weather.

### **IV. Strategic Recommendations**

_Based on the behavioral differences identified in the data, I propose the following three strategies to convert casual riders into annual members:_

#### **1. Seasonal Membership Strategy (The "Winter Gap" Fix)**
* **The Data:** Casual rider volume drops by **92.68%** in winter, while members are more resilient (only a **75.14%** drop).
* **The Recommendation:** Launch a "Pre-Season" membership drive in late February or early March. Since members are more likely to ride year-round, converting casual riders *before* the spring surge begins will stabilize annual revenue and build a consistent user base that isn't entirely dependent on perfect weather.

#### **2. Optimize for Leisure Use Cases (The "Unlimited" Value)**
* **The Data:** Casual riders average **22.59 minutes** per trip—nearly double the **12.42 minutes** of annual members. 
* **The Recommendation:** Marketing campaigns should focus on the **"unlimited ride access"** of the annual membership. For users who enjoy long-duration weekend trips, a membership removes the "per-minute" or "per-ride" transaction stress. Positioning the membership as a way to "Explore Chicago without limits" appeals directly to the high-duration behavior seen in the casual group.

#### **3. Targeted Weekend Conversion (The "Weekend Warrior")**
* **The Data:** Casual trips spike on **Saturday and Sunday**, while member usage actually decreases during the same period.
* **The Recommendation:** Target promotions specifically for the weekend at high-traffic recreational stations. Since these users clearly value the service for weekend recreation, the marketing message should frame the annual membership not as a "commuter pass," but as a **"Weekend Adventure Pass"** that pays for itself over the course of the peak summer season.


_By focusing on these data-driven strategies, Cyclistic can move away from general awareness marketing and toward a high-conversion model that maximizes long-term profitability._